In [18]:
!pip install cryptography

In [25]:
import sqlite3
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from cryptography.hazmat.primitives import serialization

# ── Step 1: Load Gold table from SQLite ────────────────────────────────────
sqlite_conn = sqlite3.connect("cat_fleet_medallion.db")
df_gold = pd.read_sql("SELECT * FROM gold_telematics", sqlite_conn)
sqlite_conn.close()
print(f"Loaded {len(df_gold)} rows, {len(df_gold.columns)} columns from SQLite Gold table")

# ── Step 2: Connect to Snowflake using key-pair auth ───────────────────────
with open(r"C:\Users\reddy\OneDrive\Desktop\cat-fleet-predictive-maintenance\rsa_key.p8", "rb") as key_file:
    private_key = serialization.load_pem_private_key(key_file.read(), password=None)

pkb = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

sf_conn = snowflake.connector.connect(
    account='TGB56921',
    user='SPT',
    private_key=pkb,
    warehouse='SPT_WH',
    database='CAT_FLEET_DB',
    schema='PUBLIC'
)
print("Connected to Snowflake")

# ── Step 3: Push Gold table to Snowflake ───────────────────────────────────
df_gold.columns = [c.upper() for c in df_gold.columns]
success, num_chunks, num_rows, output = write_pandas(
    conn=sf_conn,
    df=df_gold,
    table_name='GOLD_TELEMATICS',
    auto_create_table=True,
    overwrite=True
)
print(f"Upload success: {success}")
print(f"Rows written: {num_rows}, Chunks: {num_chunks}")

# ── Step 4: Verify by querying back ────────────────────────────────────────
cursor = sf_conn.cursor()
cursor.execute("SELECT COUNT(*) FROM GOLD_TELEMATICS")
count = cursor.fetchone()[0]
print(f"Verified row count in Snowflake: {count}")

cursor.execute("SELECT FINAL_TARGET_LABEL, COUNT(*) FROM GOLD_TELEMATICS GROUP BY FINAL_TARGET_LABEL")
for row in cursor.fetchall():
    print(row)

cursor.close()
sf_conn.close()
print("Connection closed")

Loaded 1970 rows, 56 columns from SQLite Gold table
Connected to Snowflake
Upload success: True
Rows written: 1970, Chunks: 1
Verified row count in Snowflake: 1970
('Brake', 11)
('Healthy', 1945)
('Battery', 10)
('Engine', 4)
Connection closed
